# L09 · Synthetic Data Recording and Collection Throughput

This lab turns the L08 scripted rollout into a small persistent dataset:

```text
align observation and action → sample at dataset FPS → accept or discard an episode
→ write two successful episodes → reopen and inspect the saved data
```

The shared package owns the scene, expert, recorder, and LeRobot integration. The notebook keeps the sampling decision, feature schema, success gate, writer lifecycle, and readback checks visible.

## Before you run

Install the course data dependencies before starting Jupyter:

```bash
uv sync --locked --extra data
```

`ROBO_GENESIS_BACKEND=auto` selects the verified AMD backend when available and otherwise uses CPU; use `cpu` to request CPU explicitly. `ROBO_GENESIS_RENDER` defaults to `1` because both camera streams are lesson data. Setting it to `0` runs only the sampling and schema diagnostics—it does not record a dataset or complete the experiment. Restart the kernel after changing either variable.

The output is `ROBO_GENESIS_DATASETS_DIR/l09_banana_demo`. The notebook stops if that exact directory exists. Set `RG101_L09_OVERWRITE=1` before kernel startup only when you intend to replace it.

Predict first:

1. Which control-step indices will 100 Hz → 5 FPS retain?
2. Why do 100 Hz → 30 FPS gaps alternate between three and four steps?
3. Why is an unsuccessful attempt discarded as a whole?
4. What must be checked after reopening the dataset?

In [ ]:
import importlib.metadata
import os
import shutil
from pathlib import Path

from robo_genesis.course_manifest import load_course_manifest
from robo_genesis.paths import DATASETS_DIR, OUTPUTS_DIR

lesson = load_course_manifest().lesson('L09')
assert lesson.slug == 'synthetic-data-recording-and-throughput'
assert lesson.duration_minutes == 120
assert lesson.hardware.value == 'gpu-recommended'
assert lesson.status.value == 'gpu-verified'

backend_mode = os.environ.get('ROBO_GENESIS_BACKEND', 'auto').strip().lower()
if backend_mode not in {'auto', 'cpu'}:
    raise ValueError("ROBO_GENESIS_BACKEND must be 'auto' or 'cpu'")
render_value = os.environ.get('ROBO_GENESIS_RENDER', '1').strip()
if render_value not in {'0', '1'}:
    raise ValueError("ROBO_GENESIS_RENDER must be '0' or '1'")
render_enabled = render_value == '1'
overwrite_enabled = os.environ.get('RG101_L09_OVERWRITE', '0').strip() == '1'

REPO_ID = 'local/l09_banana_demo'
CONTROL_FPS = 100
DATASET_FPS = 5
IMG_WH = (160, 120)
TARGET_EPISODES = 2
MAX_ATTEMPTS = 10

dataset_parent = DATASETS_DIR.resolve()
dataset_root = (dataset_parent / 'l09_banana_demo').resolve()
if dataset_root.parent != dataset_parent or dataset_root.name != 'l09_banana_demo':
    raise ValueError(f'Unsafe dataset path: {dataset_root}')

cache_root = (OUTPUTS_DIR / 'l09_cache').resolve()
cache_paths = {
    'HF_DATASETS_CACHE': cache_root / 'huggingface_datasets',
    'XDG_CACHE_HOME': cache_root / 'xdg',
    'MPLCONFIGDIR': cache_root / 'matplotlib',
}
for variable, path in cache_paths.items():
    os.environ.setdefault(variable, str(path))
    Path(os.environ[variable]).mkdir(parents=True, exist_ok=True)
dataset_parent.mkdir(parents=True, exist_ok=True)

if render_enabled and dataset_root.exists():
    if not overwrite_enabled:
        raise FileExistsError(
            f'{dataset_root} already exists; choose another datasets directory or set '
            'RG101_L09_OVERWRITE=1 before restarting the kernel'
        )
    shutil.rmtree(dataset_root)

required_versions = {'genesis-world': '1.3.3', 'lerobot': '0.6.0', 'av': '15.1.0'}
installed_versions = {name: importlib.metadata.version(name) for name in required_versions}
version_mismatches = {
    name: (installed_versions[name], expected)
    for name, expected in required_versions.items()
    if installed_versions[name] != expected
}
if version_mismatches:
    raise RuntimeError(f'Dependency version mismatch: {version_mismatches}')

print(f'L09: {lesson.duration_minutes} min, hardware={lesson.hardware.value}, status={lesson.status.value}')
print(f'Requested backend={backend_mode}; render={render_enabled}; overwrite={overwrite_enabled}')
print(f'Dataset root: {dataset_root}')
if not render_enabled:
    print('DIAGNOSTIC MODE: no cameras, recording, persistence, readback, or visual evidence')

## Choose complete frames at the dataset rate

The recorder is called before each simulator step, so a retained row pairs the current measured state and both current camera views with the command about to be applied. One sampling decision keeps or skips that entire group.

![One pre-step sampling decision keeps the aligned observation and action; the complete attempt is later committed or discarded.](../../docs/public/diagrams/l09-alignment-and-episode-transaction.svg)

The sampling cell reproduces `EpisodeRecorder`'s accumulator: it retains callback 0, then distributes non-integer spacing deterministically. Change `CANDIDATE_DATASET_FPS` to 10, 20, or 30 without rerunning Genesis. Before running, predict the gap pattern and retained-frame count.

In [ ]:
import numpy as np

def describe_schedule(dataset_fps, total_control_steps=CONTROL_FPS):
    # Match EpisodeRecorder.reset(): preload one interval to retain callback 0.
    steps_per_frame = CONTROL_FPS / dataset_fps
    accum = steps_per_frame
    retained = []
    for control_step in range(total_control_steps):
        accum += 1.0
        if accum < steps_per_frame:
            continue
        accum -= steps_per_frame
        retained.append(control_step)
    gaps = np.diff(retained)
    logical_timestamps = np.arange(len(retained), dtype=float) / dataset_fps
    return {
        'indices': tuple(retained),
        'gaps': tuple(sorted(set(gaps.tolist()))),
        'average_fps': len(retained) / (total_control_steps / CONTROL_FPS),
        'logical_timestamps': logical_timestamps,
    }


five_fps_schedule = describe_schedule(DATASET_FPS)
thirty_fps_schedule = describe_schedule(30)

sampling_checks = {
    'five_fps_indices': five_fps_schedule['indices'] == (0, 19, 39, 59, 79, 99),
    'five_fps_gaps': five_fps_schedule['gaps'] == (19, 20),
    'five_fps_logical_time': np.allclose(
        five_fps_schedule['logical_timestamps'], np.arange(6) / 5
    ),
    'thirty_fps_count': len(thirty_fps_schedule['indices']) == 30,
    'thirty_fps_initial_indices': thirty_fps_schedule['indices'][:5] == (0, 3, 6, 10, 13),
    'thirty_fps_gaps': thirty_fps_schedule['gaps'] == (3, 4),
}
failed_sampling = [name for name, passed in sampling_checks.items() if not passed]
if failed_sampling:
    raise AssertionError('Sampling checks failed: ' + ', '.join(failed_sampling))

for rate, schedule in ((5, five_fps_schedule), (30, thirty_fps_schedule)):
    print(
        f'{CONTROL_FPS} Hz → {rate} FPS: count={len(schedule["indices"])}, '
        f'gaps={schedule["gaps"]}, average={schedule["average_fps"]:.1f}, '
        f'first indices={schedule["indices"][:8]}, '
        f'first logical times={schedule["logical_timestamps"][:5]}'
    )

CANDIDATE_DATASET_FPS = 30
candidate_schedule = describe_schedule(CANDIDATE_DATASET_FPS)
print(
    f'Candidate {CANDIDATE_DATASET_FPS} FPS retains {len(candidate_schedule["indices"])} '
    f'frames per simulated second; wall-clock throughput has not been measured.'
)

## Inspect the four recorded features

Each retained frame contains measured Franka joint positions, the expert's nine-value position-target action, and synchronized world and wrist RGB images. The natural-language task is supplied with each frame, while LeRobot adds bookkeeping fields such as `index`, `episode_index`, `frame_index`, `timestamp`, and `task_index` when an episode is saved.

The finger action entries are `0.0` while force is used to close the real simulated fingers. In this dataset, those zeros are the policy-facing position-target proxy for ‘closed,’ not measured finger positions or force commands.

In [ ]:
from robo_genesis.record_dataset import JOINT_NAMES, build_features, task_description

USER_FEATURE_KEYS = {
    'observation.state',
    'action',
    'observation.images.world',
    'observation.images.wrist',
}
features = build_features(IMG_WH)
task_text = task_description('011_banana')
width, height = IMG_WH

feature_checks = {
    'four_user_features': set(features) == USER_FEATURE_KEYS,
    'nine_joint_names': len(JOINT_NAMES) == 9,
    'state_shape_and_names': tuple(features['observation.state']['shape']) == (9,)
    and tuple(features['observation.state']['names']) == tuple(JOINT_NAMES),
    'action_shape_and_names': tuple(features['action']['shape']) == (9,)
    and tuple(features['action']['names']) == tuple(JOINT_NAMES),
    'world_hwc_video': tuple(features['observation.images.world']['shape']) == (height, width, 3),
    'wrist_hwc_video': tuple(features['observation.images.wrist']['shape']) == (height, width, 3),
    'task_text': task_text == 'pick the banana and place it in the bowl',
}
failed_features = [name for name, passed in feature_checks.items() if not passed]
if failed_features:
    raise AssertionError('Feature checks failed: ' + ', '.join(failed_features))

for key, specification in features.items():
    print(f'{key}: dtype={specification["dtype"]}, shape={tuple(specification["shape"])}')
print('Joint order:', ', '.join(JOINT_NAMES))
print('Task:', task_text)

## Build the recorder and writer

The complete experiment initializes Genesis once, builds one environment with both cameras, and creates a local LeRobot writer. Domain randomization is disabled here so that L09 isolates recording behavior; L11 introduces it separately.

When rendering is disabled, this cell deliberately creates none of those runtime objects. That branch is useful for checking the earlier pure logic, but it is not a camera-data substitute.

In [ ]:
bundle = None
randomizer = None
recorder = None
dataset = None
actual_backend = 'not initialized'
runtime_checks = {'diagnostic_mode_requested': not render_enabled}

if render_enabled:
    import genesis as gs
    from lerobot.configs.video import RGBEncoderConfig
    from lerobot.datasets.lerobot_dataset import LeRobotDataset

    from robo_genesis.build_scene import build_scene
    from robo_genesis.course_utils import select_backend
    from robo_genesis.randomize import EnvRandomizer, RandomizationConfig
    from robo_genesis.record_dataset import EpisodeRecorder

    backend = gs.cpu if backend_mode == 'cpu' else select_backend(prefer_rocm=True)
    gs.init(backend=backend, seed=0, precision='32', logging_level='warning')
    if getattr(gs, 'amdgpu', None) is not None and gs.backend == gs.amdgpu:
        actual_backend = 'amdgpu'
    elif gs.backend == gs.cpu:
        actual_backend = 'cpu'
    else:
        actual_backend = str(gs.backend)

    bundle = build_scene(
        show_viewer=False,
        n_envs=1,
        add_world_cam=True,
        add_wrist_cam=True,
        add_video_cam=False,
        draw_world_frame=False,
        scene_dr=None,
    )
    randomizer = EnvRandomizer(
        bundle,
        RandomizationConfig(randomize_pick=False, randomize_place=False, seed=0),
    )
    dataset = LeRobotDataset.create(
        repo_id=REPO_ID,
        root=dataset_root,
        fps=DATASET_FPS,
        features=features,
        robot_type='franka',
        use_videos=True,
        rgb_encoder=RGBEncoderConfig(vcodec='h264', video_backend='pyav'),
    )
    recorder = EpisodeRecorder(
        bundle,
        fps=DATASET_FPS,
        img_wh=IMG_WH,
        control_fps=CONTROL_FPS,
    )
    runtime_checks = {
        'supported_backend': actual_backend in {'cpu', 'amdgpu'},
        'explicit_cpu_honored': backend_mode != 'cpu' or actual_backend == 'cpu',
        'world_camera_ready': bundle.world_cam is not None,
        'wrist_camera_ready': bundle.wrist_cam is not None,
        'serial_scene': bundle.scene.n_envs in {0, 1},
        'writer_root': dataset.root.resolve() == dataset_root,
        'recorder_rate': recorder.fps == DATASET_FPS,
    }
    failed_runtime = [name for name, passed in runtime_checks.items() if not passed]
    if failed_runtime:
        raise AssertionError('Runtime setup checks failed: ' + ', '.join(failed_runtime))
    print(
        f'Runtime ready: backend={actual_backend} (requested={backend_mode}), '
        f'cameras=world+wrist, writer={dataset.root}'
    )
else:
    print('SKIP — rendering disabled; scene, cameras, recorder, and dataset writer were not created')

## Record complete attempts, then keep successes

Every attempt gets a fresh scene reset, recorder buffer, and sampling phase. Only after release and settling does `success` decide the transaction: a successful non-empty buffer is flushed and closed with `save_episode()`; a failed buffer is left out of the writer. After two accepted episodes—or ten attempts—the dataset is closed once with `finalize()`.

In [ ]:
import time

from robo_genesis.grasp_demo import check_success, run_pick_place

attempt_summaries = []
accepted_frame_counts = []
attempts = 0
recording_wall_seconds = 0.0
dataset_finalized = False
recording_checks = {'diagnostic_mode_requested': not render_enabled}

if render_enabled:
    collection_started = time.perf_counter()
    try:
        while len(accepted_frame_counts) < TARGET_EPISODES and attempts < MAX_ATTEMPTS:
            episode_seed = attempts
            attempts += 1
            task = randomizer.reset(seed=episode_seed)
            if task.pick_object != '011_banana' or task.place_target != '024_bowl':
                raise AssertionError(f'Unexpected task: {task}')

            recorder.reset()
            success, _ = run_pick_place(bundle, task, recorder=recorder)
            confirmed_success = check_success(bundle, task)

            buffer_lengths = (
                len(recorder.states),
                len(recorder.actions),
                len(recorder.world_imgs),
                len(recorder.wrist_imgs),
            )
            states = np.stack(recorder.states)
            actions = np.stack(recorder.actions)
            world_images = np.stack(recorder.world_imgs)
            wrist_images = np.stack(recorder.wrist_imgs)
            attempt_checks = {
                'success_predicate_agrees': bool(success) == bool(confirmed_success),
                'equal_nonzero_buffers': len(set(buffer_lengths)) == 1 and buffer_lengths[0] > 0,
                'state_shape': states.shape == (len(recorder), 9),
                'action_shape': actions.shape == (len(recorder), 9),
                'state_action_floating': np.issubdtype(states.dtype, np.floating)
                and np.issubdtype(actions.dtype, np.floating),
                'state_action_finite': np.isfinite(states).all() and np.isfinite(actions).all(),
                'world_hwc_uint8': world_images.shape == (len(recorder), height, width, 3)
                and world_images.dtype == np.uint8,
                'wrist_hwc_uint8': wrist_images.shape == (len(recorder), height, width, 3)
                and wrist_images.dtype == np.uint8,
            }
            failed_attempt_checks = [
                name for name, passed in attempt_checks.items() if not passed
            ]
            if failed_attempt_checks:
                raise AssertionError(
                    f'Attempt {attempts} checks failed: ' + ', '.join(failed_attempt_checks)
                )

            decision = 'discarded'
            if success and len(recorder) > 0:
                recorder.flush_to(dataset, task_text)
                accepted_frame_counts.append(len(recorder))
                decision = 'accepted'

            attempt_summaries.append(
                {
                    'attempt': attempts,
                    'seed': episode_seed,
                    'success': bool(success),
                    'sampled_frames': len(recorder),
                    'decision': decision,
                }
            )
            summary = attempt_summaries[-1]
            print(
                f'Attempt {summary["attempt"]}: success={summary["success"]} → '
                f'{summary["decision"]}; dataset frames={summary["sampled_frames"]}'
            )
    finally:
        dataset.finalize()
        dataset_finalized = True
        recording_wall_seconds = time.perf_counter() - collection_started

    recording_checks = {
        'target_episode_count': len(accepted_frame_counts) == TARGET_EPISODES,
        'attempt_limit_honored': attempts <= MAX_ATTEMPTS,
        'positive_frame_counts': all(count > 0 for count in accepted_frame_counts),
        'dataset_finalized': dataset_finalized,
    }
    failed_recording = [name for name, passed in recording_checks.items() if not passed]
    if failed_recording:
        raise AssertionError('Recording checks failed: ' + ', '.join(failed_recording))
    print(
        f'Recorded {len(accepted_frame_counts)} accepted episodes in {attempts} attempts; '
        f'frames per accepted episode={accepted_frame_counts}'
    )
else:
    print('SKIP — no attempts were run and no dataset was written')

## Reopen what was actually written

Fresh metadata and reader objects test persistence rather than reusing the writer's in-memory state. The checks below verify episode boundaries, logical timestamps, joint order, task text, and PyAV decoding for both cameras.

In [ ]:
metadata = None
recorded = None
readback_checks = {'diagnostic_mode_requested': not render_enabled}

if render_enabled:
    import torch
    from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata

    metadata = LeRobotDatasetMetadata(REPO_ID, root=dataset_root)
    recorded = LeRobotDataset(REPO_ID, root=dataset_root, video_backend='pyav')
    rows = recorded.hf_dataset
    plain_rows = rows.with_format(None)

    global_indices = np.asarray(plain_rows['index'], dtype=np.int64)
    episode_indices = np.asarray(plain_rows['episode_index'], dtype=np.int64)
    frame_indices = np.asarray(plain_rows['frame_index'], dtype=np.int64)
    timestamps = np.asarray(plain_rows['timestamp'], dtype=float)
    expected_total_frames = sum(accepted_frame_counts)
    episode_starts = [
        int(np.flatnonzero(episode_indices == episode)[0])
        for episode in range(TARGET_EPISODES)
    ]
    representative_samples = [recorded[index] for index in episode_starts]

    expected_features = USER_FEATURE_KEYS | {
        'index', 'episode_index', 'frame_index', 'timestamp', 'task_index'
    }
    episode_boundaries_ok = all(
        np.array_equal(
            frame_indices[episode_indices == episode],
            np.arange(np.sum(episode_indices == episode)),
        )
        and np.allclose(
            timestamps[episode_indices == episode],
            frame_indices[episode_indices == episode] / DATASET_FPS,
        )
        for episode in range(TARGET_EPISODES)
    )
    decoded_values_ok = all(
        sample['observation.state'].shape == (9,)
        and sample['action'].shape == (9,)
        and sample['observation.state'].dtype == torch.float32
        and sample['action'].dtype == torch.float32
        and torch.isfinite(sample['observation.state']).all().item()
        and torch.isfinite(sample['action']).all().item()
        and sample['observation.images.world'].shape == (3, height, width)
        and sample['observation.images.wrist'].shape == (3, height, width)
        and sample['observation.images.world'].dtype == torch.float32
        and sample['observation.images.wrist'].dtype == torch.float32
        and torch.isfinite(sample['observation.images.world']).all().item()
        and torch.isfinite(sample['observation.images.wrist']).all().item()
        and sample['task'] == task_text
        for sample in representative_samples
    )
    persistence_paths_ok = (dataset_root / 'meta' / 'info.json').is_file()
    persistence_paths_ok &= (dataset_root / 'meta' / 'tasks.parquet').is_file()
    persistence_paths_ok &= any((dataset_root / 'meta' / 'episodes').rglob('*.parquet'))
    persistence_paths_ok &= any((dataset_root / 'data').rglob('*.parquet'))
    persistence_paths_ok &= any((dataset_root / 'videos').rglob('*.mp4'))

    readback_checks = {
        'metadata_counts': metadata.fps == DATASET_FPS
        and metadata.total_episodes == TARGET_EPISODES
        and metadata.total_tasks == 1
        and metadata.total_frames == expected_total_frames > 0,
        'feature_set': set(metadata.features) == expected_features,
        'joint_names': tuple(metadata.features['observation.state']['names']) == tuple(JOINT_NAMES)
        and tuple(metadata.features['action']['names']) == tuple(JOINT_NAMES),
        'global_index_continuity': np.array_equal(
            global_indices, np.arange(expected_total_frames)
        ),
        'episode_index_monotonic': np.all(np.diff(episode_indices) >= 0),
        'episode_boundaries': episode_boundaries_ok,
        'decoded_values': decoded_values_ok,
        'persistent_files': persistence_paths_ok,
    }
    failed_readback = [name for name, passed in readback_checks.items() if not passed]
    if failed_readback:
        raise AssertionError('Readback checks failed: ' + ', '.join(failed_readback))
    print(
        f'Read back {metadata.total_frames} frames, {metadata.total_episodes} episodes, '
        f'{metadata.total_tasks} task; both camera streams decoded through PyAV.'
    )
else:
    print('SKIP — no persisted dataset exists to reopen in diagnostic mode')

## Inspect boundaries, commands, states, and camera views

All plots now come from the reopened dataset. The timeline should show a continuous global index and episode-local frame/time resets. Arm and finger values use separate panels because their units differ. The montage decodes start, middle, and end from one persisted episode, with world and wrist views from the same rows.

In [ ]:
visual_checks = {'diagnostic_mode_requested': not render_enabled}

if render_enabled:
    import matplotlib.pyplot as plt

    state_rows = np.asarray(plain_rows['observation.state'], dtype=float)
    action_rows = np.asarray(plain_rows['action'], dtype=float)

    timeline, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    axes[0].step(global_indices, episode_indices, where='post')
    axes[0].set_ylabel('episode_index')
    axes[0].set_title('Global rows continue while episode identity changes')
    axes[1].plot(global_indices, frame_indices, label='frame_index')
    axes[1].plot(global_indices, timestamps, label='timestamp (s)')
    axes[1].set_xlabel('global index')
    axes[1].set_title('Frame index and logical time restart for each episode')
    axes[1].legend()
    timeline.tight_layout()
    plt.show()

    arm_error = np.abs(action_rows[:, :7] - state_rows[:, :7])
    tracking_joint = int(np.argmax(np.max(arm_error, axis=0)))
    traces, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    axes[0].plot(action_rows[:, tracking_joint], '--', label='commanded target')
    axes[0].plot(state_rows[:, tracking_joint], label='measured qpos')
    axes[0].set_ylabel('arm angle (rad)')
    axes[0].set_title(f'Arm joint {tracking_joint}: command and measured state')
    axes[0].legend()
    for finger in range(2):
        axes[1].plot(action_rows[:, 7 + finger], '--', label=f'finger {finger} target')
        axes[1].plot(state_rows[:, 7 + finger], label=f'finger {finger} measured')
    axes[1].set_xlabel('global index')
    axes[1].set_ylabel('finger position (m)')
    axes[1].set_title('Finger proxy targets and measured positions')
    axes[1].legend(ncol=2)
    traces.tight_layout()
    plt.show()

    first_episode_rows = np.flatnonzero(episode_indices == 0)
    selected_rows = first_episode_rows[[0, len(first_episode_rows) // 2, -1]]
    selected_samples = [recorded[int(index)] for index in selected_rows]
    image_keys = ('observation.images.world', 'observation.images.wrist')
    decoded_images = {
        key: [sample[key].detach().cpu().numpy().transpose(1, 2, 0) for sample in selected_samples]
        for key in image_keys
    }
    visual_checks = {
        'state_action_rows': state_rows.shape == action_rows.shape == (metadata.total_frames, 9),
        'state_action_finite': np.isfinite(state_rows).all() and np.isfinite(action_rows).all(),
        'six_decoded_images': sum(len(images) for images in decoded_images.values()) == 6,
        'image_shapes': all(
            image.shape == (height, width, 3)
            for images in decoded_images.values()
            for image in images
        ),
        'nonempty_images': all(
            np.isfinite(image).all() and float(np.std(image)) > 0.0
            for images in decoded_images.values()
            for image in images
        ),
    }
    failed_visual = [name for name, passed in visual_checks.items() if not passed]
    if failed_visual:
        raise AssertionError('Visual checks failed: ' + ', '.join(failed_visual))

    montage, axes = plt.subplots(2, 3, figsize=(12, 6))
    for row, key in enumerate(image_keys):
        camera_name = key.rsplit('.', 1)[-1]
        for column, (dataset_index, sample, image) in enumerate(
            zip(selected_rows, selected_samples, decoded_images[key], strict=True)
        ):
            axes[row, column].imshow(np.clip(image, 0.0, 1.0))
            axes[row, column].set_title(
                f'{camera_name}: ep={int(sample["episode_index"])}, '
                f'frame={int(sample["frame_index"])}, t={float(sample["timestamp"]):.2f}s'
            )
            axes[row, column].axis('off')
    montage.suptitle('Start, middle, and end decoded from persisted episode 0')
    montage.tight_layout()
    plt.show()
else:
    print('SKIP — visual evidence requires the persisted camera dataset')

## Separate dataset rate from measured collection speed

For a complete run, the final cell reports committed frames per real second and accepted episodes per real hour over the attempt loop and final write. These measurements describe this run and configuration only; `DATASET_FPS=5` does not promise five written frames each wall-clock second. The notebook does not count every simulator step performed during reset and settling, so it does not report a simulated-seconds ratio.

Diagnostic mode checks only timing and schema. Its result deliberately does not use the complete-experiment success message.

In [ ]:
final_checks = {
    'manifest_contract': lesson.status.value == 'gpu-verified'
    and lesson.duration_minutes == 120,
    'dependency_contract': not version_mismatches,
    'sampling_contract': all(sampling_checks.values()),
    'feature_contract': all(feature_checks.values()),
}

if render_enabled:
    final_checks.update(
        {
            'runtime_contract': all(runtime_checks.values()),
            'recording_contract': all(recording_checks.values()),
            'readback_contract': all(readback_checks.values()),
            'visual_contract': all(visual_checks.values()),
        }
    )
else:
    final_checks['diagnostic_boundary'] = (
        bundle is None and recorder is None and dataset is None and recorded is None
    )

failed = [name for name, passed in final_checks.items() if not passed]
if failed:
    raise AssertionError('L09 checks failed: ' + ', '.join(failed))

if render_enabled:
    dataset_bytes = sum(path.stat().st_size for path in dataset_root.rglob('*') if path.is_file())
    committed_frames_per_second = metadata.total_frames / recording_wall_seconds
    accepted_episodes_per_hour = len(accepted_frame_counts) * 3600.0 / recording_wall_seconds
    print(
        f'Observed throughput for attempt loop + finalize ({actual_backend}, n_envs=1, '
        f'{DATASET_FPS} FPS, {width}×{height}, h264; failed attempts included): '
        f'{committed_frames_per_second:.2f} committed frames/s, '
        f'{accepted_episodes_per_hour:.2f} accepted episodes/h'
    )
    print(
        f'Run evidence: attempts={attempts}, accepted_frames={accepted_frame_counts}, '
        f'elapsed={recording_wall_seconds:.2f}s, dataset_size={dataset_bytes / 1_000_000:.2f} MB'
    )
    print('L09 CHECK: PASSED')
else:
    print('L09 DIAGNOSTIC CHECK: PASSED — sampling and schema only')
    print('Core recording experiment: NOT COMPLETED (camera rendering is disabled)')

## Checkpoint and connection to L10

Using the schedule, saved metadata, plots, and montage, explain:

- why recording `(state_{t+1}, action_t)` would be a one-frame alignment error;
- why 30 FPS from a 100 Hz controller needs both three- and four-step gaps;
- why state, action, and both images share one sampling clock;
- why `save_episode()` commits one accepted episode while `finalize()` closes the whole dataset;
- why multiple simulator environments alone do not guarantee a correct or faster parallel recorder; and
- what two successful episodes prove—and what they do not say about expert reliability, diversity, or training quality.

L10 will start from these persisted fields and explain how an imitation-learning model turns observation/action rows into training inputs and targets.